# HDI Predictor - Data Analysis & Model Training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## 1. Load & Inspect Dataset

In [ ]:
df = pd.read_csv("../data/hdi_dataset.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 2. Handle Missing Values

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
df.isnull().sum()

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['HDI'], kde=True, bins=15)
plt.title("Distribution of HDI Scores")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.scatterplot(x='Life_Expectancy', y='HDI', data=df, ax=axes[0])
sns.scatterplot(x='Mean_Years_Schooling', y='HDI', data=df, ax=axes[1])
sns.scatterplot(x='GNI_per_capita', y='HDI', data=df, ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
sns.stripplot(x=pd.cut(df['GNI_per_capita'], bins=5), y=df['HDI'])
plt.xticks(rotation=45)
plt.title("GNI per Capita (binned) vs HDI")
plt.show()

## 4. Feature Selection & Train/Test Split

In [ ]:
X = df[['Life_Expectancy', 'Mean_Years_Schooling',
        'Expected_Years_Schooling', 'GNI_per_capita']]
y = df['HDI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 5. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

## 7. Evaluate Model

In [ ]:
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2 Score: {r2:.4f}")

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual HDI")
plt.ylabel("Predicted HDI")
plt.title("Actual vs Predicted HDI")
plt.show()

## 8. Save Model & Scaler with Pickle

In [ ]:
with open("../model/hdi_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("../model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved.")

## 9. Classification Helper & Scenario Testing

In [ ]:
def classify_hdi(score):
    if score >= 0.800:
        return "Very High Human Development"
    elif score >= 0.700:
        return "High Human Development"
    elif score >= 0.550:
        return "Medium Human Development"
    else:
        return "Low Human Development"

test_cases = [
    {"Life_Expectancy": 82, "Mean_Years_Schooling": 13.5, "Expected_Years_Schooling": 17, "GNI_per_capita": 55000},
    {"Life_Expectancy": 68, "Mean_Years_Schooling": 7.5, "Expected_Years_Schooling": 11, "GNI_per_capita": 8000},
    {"Life_Expectancy": 58, "Mean_Years_Schooling": 4, "Expected_Years_Schooling": 7, "GNI_per_capita": 1800},
]

for case in test_cases:
    row = pd.DataFrame([case])
    row_scaled = scaler.transform(row)
    pred = model.predict(row_scaled)[0]
    print(f"{case} -> HDI: {pred:.3f} ({classify_hdi(pred)})")